In [ ]:
import xarray as xr
import rioxarray
import numpy as np
import glob
import re
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from utils import retrieve_url
from data_funcs import int2fstep

## Retrieve Files

In [ ]:
bands = [616, 620]
hrs = ["00", "01", "02"] # Hour of day 00-23
fstep = int2fstep(0)
doy_str = 
base_url = f"https://demo.openwfm.org/web/data/fmda//tif/{doy_str}/"

In [ ]:
for band in bands:
    for hr in hrs:
        base_filename = f"hrrr.t{hr}z.wrfprs{fstep}"
        filename = f"{base_filename}.{band}.tif"
        retrieve_url(
            f"{base_url}/{filename}",
            dest_path = f"{filename}"
        )

In [ ]:
ds1 = xr.open_dataset(f"{base_filename}.616.tif")
ds2 = xr.open_dataset(f"{base_filename}.620.tif")

In [ ]:
type(ds1)

In [ ]:
ds1.band_data.shape

In [ ]:
print(ds1.band_data.dims)
print(ds1.band_data.coords)

In [ ]:
ds2.band_data.shape

In [ ]:
temp = ds1.band_data
rh = ds2.band_data

In [ ]:
type(temp)

In [ ]:
Ed = 0.924*rh**0.679 + 0.000499*np.exp(0.1*rh) + 0.18*(21.1 + 273.15 - temp)*(1 - np.exp(-0.115*rh))
Ew = 0.618*rh**0.753 + 0.000454*np.exp(0.1*rh) + 0.18*(21.1 + 273.15 - temp)*(1 - np.exp(-0.115*rh)) 

In [ ]:
type(Ed)

In [ ]:
Ed.dims

In [ ]:
plt.imshow(Ed.isel(band=0))

In [ ]:
type(Ed)

## Open Multiple Datasets

In [ ]:
file_list = [f"{base_filename}.{band}.tif" for band in bands]
file_list

In [ ]:
data = xr.open_mfdataset(file_list, concat_dim="band", combine="nested")

In [ ]:
data

In [ ]:
data.band_data.shape

In [ ]:
band_df_hrrr = pd.DataFrame({
    'Band': [616, 620, 624, 628, 629, 661, 561, 612, 643],
    'hrrr_name': ['TMP', 'RH', "WIND", 'PRATE', 'APCP',
                  'DSWRF', 'SOILW', 'CNWAT', 'GFLUX'],
    'dict_name': ["temp", "rh", "wind", "rain", "precip_accum",
                 "solar", "soilm", "canopyw", "groundflux"],
    'descr': ['2m Temperature [K]', 
              '2m Relative Humidity [%]', 
              '10m Wind Speed [m/s]'
              'surface Precip. Rate [kg/m^2/s]',
              'surface Total Precipitation [kg/m^2]',
              'surface Downward Short-Wave Radiation Flux [W/m^2]',
              'surface Total Precipitation [kg/m^2]',
              '0.0m below ground Volumetric Soil Moisture Content [Fraction]',
              'Plant Canopy Surface Water [kg/m^2]',
              'surface Ground Heat Flux [W/m^2]']
})

In [ ]:
def bands_to_names(bands, band_df):
    # Get the bands from file names
    # bands = band_from_hrrrname(file_list)
    # Find matching dict_name values in the dataframe
    dict_names = band_df.loc[band_df['Band'].isin(bands), 'dict_name'].tolist()
    return dict_names

In [ ]:
bands_to_names(bands, band_df_hrrr)

In [ ]:
band_names = bands_to_names(bands, band_df_hrrr)
data = data.assign_coords(band = ("band", band_names))

In [ ]:
data.band

In [ ]:
def calc_eqs(ds):

    # Calculate Ed based on temp and rh
    temp = ds.sel(band="temp")
    rh = ds.sel(band="rh")
    
    Ed = 0.924 * rh**0.679 + 0.000499 * np.exp(0.1 * rh) + 0.18 * (21.1 + 273.15 - temp) * (1 - np.exp(-0.115 * rh))
    Ew = 0.618 * rh**0.753 + 0.000454 * np.exp(0.1 * rh) + 0.18 * (21.1 + 273.15 - temp) * (1 - np.exp(-0.115 * rh))
    
    # Expand dims and assign new band names for Ed and Ew
    Ed = Ed.expand_dims(dim="band").assign_coords(band=["Ed"])
    Ew = Ew.expand_dims(dim="band").assign_coords(band=["Ew"])

    ds = xr.concat([ds, Ed, Ew], dim="band")
    
    return ds
    

In [ ]:
data = calc_eqs(data)

In [ ]:
data

In [ ]:
data.band

In [ ]:
features_list = ['Ed', 'Ew']

In [ ]:
subset = data.sel(band=features_list)

In [ ]:
subset.band_data.shape

In [ ]:
subset

In [ ]:
subset.dims

In [ ]:
subset.band_data.shape

In [ ]:
plt.imshow(subset.sel(band="Ed")['band_data'])

In [ ]:
file_list = ['hrrr.t00z.wrfprsf00.616.tif', 'hrrr.t00z.wrfprsf00.620.tif', 'hrrr.t01z.wrfprsf00.616.tif', 'hrrr.t01z.wrfprsf00.620.tif', 'hrrr.t02z.wrfprsf00.616.tif', 'hrrr.t02z.wrfprsf00.620.tif']
file_list

In [ ]:
unique_times = sorted(set(re.search(r't\d{2}z', f).group() for f in file_list))
unique_times

In [ ]:
grouped_files = [[f for f in file_list if time in f] for time in unique_times]
grouped_files

In [ ]:
# Preprocess function to extract time information from filename and set it as a coordinate
def preprocess(ds):
    # Extract the time (e.g., "t00z") from the filename
    time_str = re.search(r't\d{2}z', ds.encoding['source']).group()
    ds = ds.assign_coords(time=time_str)  # Add time coordinate
    return ds

In [ ]:
data = xr.open_mfdataset(
    grouped_files,
    concat_dim=["time", "band"],
    combine="nested",
    preprocess=preprocess
)

In [ ]:
data

In [ ]:
data.time